In [ ]:
import pickle
import tempfile
import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns

In [ ]:
file_gam = '../data/params/GAM_curves_controls.pkl'
with open(file_gam, 'rb') as handle:
    controls_GAM = pickle.load(handle)

gam_features = ['.combat.label-avg.hippunfold_volume', '.combat.label-avg.thickness.sm1', '.combat.label-avg.gyrification.sm1', '.combat.label-avg.curvature.sm1', '.combat.label-avg.gauss-curv_filtered_sm1']
sex_dict = {'male': 0, 'female': 1}



In [ ]:
sites = ['H0', 'testA', 'testB']
n_subjects = [1000, 50, 50]

np.random.seed(41)

ages = np.random.randint(5, 50, size=np.sum(n_subjects))
sexes = np.random.choice(['male', 'female'], size=np.sum(n_subjects))

sim_data = []
# for each feature, simulate data for 1000 subjects, with age range 20-80, and sex ratio 1:1, using the GAM curves for controls.
for feature in gam_features:
    for site_code, n in zip(sites, n_subjects):
        age_range = controls_GAM[feature][sex_dict['male']]['age_range']
        idx = np.argmin(np.abs(age_range - 20))
        p025 = controls_GAM[feature][sex_dict['male']]['predict_vals_intervals'][0.25][idx]
        p075 = controls_GAM[feature][sex_dict['male']]['predict_vals_intervals'][0.75][idx]
        site_offset = np.random.normal(0, 1.0 * (p075 - p025))  # add some site-specific offset to the values, drawn from a normal distribution with mean 0 and std 0.1 * (p075 - p025) to ensure that the offset is small compared to the variability in the data

        if site_code == 'H0':
            site_offset = 0  # no offset for the original site

        site_offset = 0

        for i_subject in range(n):
            age = ages[i_subject]
            sex = sexes[i_subject]

            age_range = controls_GAM[feature][sex_dict[sex]]['age_range']
            idx = np.argmin(np.abs(age_range - age))
            p005 = controls_GAM[feature][sex_dict[sex]]['predict_vals_intervals'][0.05][idx]   
            p025 = controls_GAM[feature][sex_dict[sex]]['predict_vals_intervals'][0.25][idx]   
            p050 = controls_GAM[feature][sex_dict[sex]]['predict_vals_intervals'][0.50][idx]   
            p075 = controls_GAM[feature][sex_dict[sex]]['predict_vals_intervals'][0.75][idx]   
            p095 = controls_GAM[feature][sex_dict[sex]]['predict_vals_intervals'][0.95][idx]   

            # draw from a normal distribution with parameters so that mean is p050 and std is (p075 - p025) / 1.349 (to match the interquartile range of a normal distribution)
            mean = p050
            std = (p075 - p025) / 1.349

            for hemi in ['lh', 'rh']:
                value = np.random.normal(mean, std)
                value += site_offset

                sim_data.append({'subj_id': f'sub-{i_subject+1:04d}',
                                'site': site_code, 
                                'age': age, 
                                'sex': sex, 
                                'feature': feature, 
                                'hemi': hemi,
                                'harmo': 'noharmo',
                                'value': value})

sim_data = pd.DataFrame(sim_data)

sim_data

In [ ]:
# plot sim_data on GAM curves for controls, with age on x-axis and value on y-axis

from scripts.new_patient_pipeline.run_pipeline_prediction import plot_controls_chart

for feature in gam_features:
    fig, axs = plt.subplots(1,2, figsize=(12,4), constrained_layout=True)
    for i, sex in enumerate(['male', 'female']):
        plot_controls_chart(axs[i], controls_GAM[feature][sex_dict[sex]], feature,  cmap='Greens_r', fill_color=True)
        sns.scatterplot(data=sim_data[(sim_data['feature'] == feature) & (sim_data['sex'] == sex)], 
                        x='age', 
                        y='value', 
                        hue='site',
                        ax=axs[i], 
                        color='blue', 
                        s=10,
                        alpha=0.5)
    

In [ ]:
import aidhs.distributedCombat as dc

tmpdir = tempfile.mkdtemp()

for site_code in [s for s in sites if s != 'H0']:
    for feature in gam_features:
        selected_data = sim_data[(sim_data['site'] == site_code) & (sim_data['feature'] == feature)]
        # pivot by hemi and stack into 2D array with shape (n_subjects, 2)
        selected_data = selected_data.pivot(index=['subj_id', 'site', 'age', 'sex'], columns='hemi', values='value')
        precombat_features = selected_data.values

        new_site_covars = selected_data.reset_index()[['site', 'age', 'sex']].copy()
        new_site_covars.rename(columns={'site': 'site_scanner', 'age': 'ages', 'sex': 'sex'}, inplace=True)
        # replace male with 0 and female with 1
        new_site_covars['sex'] = new_site_covars['sex'].map(sex_dict)
        new_site_covars['group'] = 0  # all controls


        site_combat_path = os.path.join(tmpdir,f'AIDHS_{site_code}','distributed_combat')
        os.makedirs(site_combat_path, exist_ok=True)
        aidhs_combat_path = '/data/params/distributed_combat/'

        feature = feature.replace('.combat', '')

        ############ 
        #check site_scanner codes are the same for all subjects
        if len(new_site_covars['site_scanner'].unique())==1:
            site_scanner = new_site_covars['site_scanner'].unique()[0]
        else:
            raise ValueError('Subjects on the list come from different site or scanner.\
            Make sure all your subject come from same site and scanner for the harmonisation process')
        bat = pd.Series(pd.Categorical(np.array(new_site_covars['site_scanner']),
                                        categories=['H0', site_scanner]))       
        # apply distributed combat
        print('step1')
        new_site_data = np.array(precombat_features).T 
        dc.distributedCombat_site(new_site_data,
                                    bat, 
                                    new_site_covars[['ages','sex','group']], 
                                    file=os.path.join(site_combat_path,f"{site_code}_{feature}_summary.pickle"), 
                                ref_batch = 'H0', 
                                robust=True,)
        print('step2')
        dc_out = dc.distributedCombat_central(
            [os.path.join(aidhs_combat_path,f'combat_{feature}.pickle'),
             os.path.join(site_combat_path,f"{site_code}_{feature}_summary.pickle")], ref_batch = 'H0'
        )
        # third, use variance estimates from full AIDHS cohort
        dc_out['var_pooled'] = pd.read_pickle(os.path.join(aidhs_combat_path,f'combat_{feature}_var.pickle')).ravel()
        for c in ['ages','sex','group']:
            new_site_covars[c]=new_site_covars[c].astype(np.float64)      
        print('step3')
        pickle_file = os.path.join(site_combat_path,f"{site_code}_{feature}_harmonisation_params_test.pickle")
        _=dc.distributedCombat_site(
            pd.DataFrame(new_site_data), bat, new_site_covars[['ages','sex','group']], 
            file=pickle_file,
                central_out=dc_out, 
            ref_batch = 'H0', 
            robust=True,
        )
        #open pickle, shrink estimates and save in hdf5 and delete pickle
        with open(pickle_file, 'rb') as f:
            params = pickle.load(f)
        #filter name keys
        target_dict = {'batch':'batches', 'delta_star':'delta.star', 'var_pooled':'var.pooled',
            'gamma_star':'gamma.star', 'stand_mean':'stand.mean', 'mod_mean': 'mod.mean', 
            'parametric': 'del', 'eb':'del', 'mean_only':'del', 'mod':'del', 'ref_batch':'del', 'beta_hat':'del', 
            }
        estimates = params['estimates'].copy()
        for key in target_dict.keys():  
            if target_dict[key]=='del':
                estimates.pop(key)
            else:
                estimates[target_dict[key]] = estimates.pop(key)
        for key in estimates.keys():
            if key in ['a_prior', 'b_prior', 't2', 'gamma_bar']:
                estimates[key]=[estimates[key]]
            if key == 'batches':
                estimates[key]=np.array([estimates[key][0]]).astype('object')
            if key=='var.pooled':
                estimates[key]=estimates[key][:,np.newaxis]
            if key in ['gamma.star', 'delta.star']:
                estimates[key]=estimates[key][np.newaxis,:]
            estimates[key] = np.array(estimates[key])
        #shrink estimates
        import aidhs.data_preprocessing
        shrink_estimates = aidhs.data_preprocessing.Preprocess.shrink_combat_estimates(None, estimates)
        combat_params_file=os.path.join(tmpdir, f'{site_code}_{feature}_combat_params.hdf5')
        aidhs.data_preprocessing.Preprocess.save_norm_combat_parameters(None, feature, shrink_estimates, combat_params_file)
        os.remove(pickle_file)
        pickle_file = os.path.join(site_combat_path,f"{site_code}_{feature}_summary.pickle")
        os.remove(pickle_file)

        #########################
        # now apply
        from neuroCombat import neuroCombatFromTraining
        combat_estimates = aidhs.data_preprocessing.Preprocess.read_norm_combat_parameters(None, feature, combat_params_file)
        combat_estimates = aidhs.data_preprocessing.Preprocess.unshrink_combat_estimates(None, combat_estimates)
        combat_estimates["batches"] = [x.split('_')[0] for x in combat_estimates["batches"]] # remove scanner strenght from the batch code if exist
        precombat_features = np.array(precombat_features)
        site_scanner = new_site_covars['site_scanner'].values
        dict_combat = neuroCombatFromTraining(dat=precombat_features.T,
                                              batch=site_scanner,
                                              estimates=combat_estimates)

        # add combat values to sim_data with harmo column as 'combat'
        harmo_data = dict_combat['data'].T
        harmo_data = pd.DataFrame(harmo_data, columns=['lh', 'rh'])
        harmo_data['subj_id'] = selected_data.reset_index()['subj_id']
        harmo_data['site'] = selected_data.reset_index()['site']
        harmo_data['age'] = selected_data.reset_index()['age']
        harmo_data['sex'] = selected_data.reset_index()['sex']
        harmo_data['feature'] = f'.combat{feature}'
        harmo_data['harmo'] = 'combat'
        harmo_data = harmo_data.melt(id_vars = ['subj_id', 'site', 'age', 'sex', 'feature', 'harmo'], var_name='hemi', value_name='value')

        sim_data = pd.concat([sim_data, harmo_data], ignore_index=True)
        


In [ ]:
# harmonize with neurocombat and use H0 as reference
from neuroCombat import neuroCombat


for feature in gam_features:
    precombat_features = sim_data[(sim_data['feature'] == feature) & (sim_data['harmo'] == 'noharmo')]
    precombat_features = precombat_features.pivot(index=['subj_id', 'site', 'age', 'sex'], columns='hemi', values='value')
    covars = precombat_features.reset_index()[['site', 'age', 'sex']].copy()
    covars.rename(columns={'site': 'site_scanner', 'age': 'ages', 'sex': 'sex'}, inplace=True)
    covars['sex'] = covars['sex'].map(sex_dict)
    covars['group'] = 0  # all controls

    dict_combat = neuroCombat(
        precombat_features.T,
        covars,
        batch_col="site_scanner",
        categorical_cols=["sex", "group"],
        continuous_cols="ages",
    )
    
    data = dict_combat["data"].T
    data = pd.DataFrame(data, columns=['lh', 'rh'])
    data['subj_id'] = precombat_features.reset_index()['subj_id']
    data['site'] = precombat_features.reset_index()['site']
    data['age'] = precombat_features.reset_index()['age']
    data['sex'] = precombat_features.reset_index()['sex']
    data['feature'] = feature
    data['harmo'] = 'combat_all'
    data = data.melt(id_vars = ['subj_id', 'site', 'age', 'sex', 'feature', 'harmo'], var_name='hemi', value_name='value')
    sim_data = pd.concat([sim_data, data], ignore_index=True)

    pass



In [ ]:
# plot sim_data on GAM curves for controls, with age on x-axis and value on y-axis
import itertools
from scripts.new_patient_pipeline.run_pipeline_prediction import plot_controls_chart

for feature, sex in itertools.product(gam_features, ['male', 'female']):
    fig, axs = plt.subplots(1,2, figsize=(12,4), constrained_layout=True)
    for i, selected_harmo in enumerate(['combat', 'combat_all']):
        plot_controls_chart(axs[i], controls_GAM[feature][sex_dict[sex]], feature,  cmap='Greens_r', fill_color=True)
        sns.scatterplot(data=sim_data[((sim_data['site'] != 'H0') &
                                       (sim_data['feature'] == feature) & 
                                       (sim_data['sex'] == sex) & 
                                       (sim_data['harmo'] == selected_harmo))], 
                        x='age', 
                        y='value', 
                        hue='site',
                        ax=axs[i], 
                        color='blue', 
                        s=10,
                        alpha=0.5)
        axs[i].set_ylim(axs[0].get_ylim())
        
    fig, axs = plt.subplots(1,2, figsize=(12,4), constrained_layout=True)
    for i, selected_harmo in enumerate(['combat', 'combat_all']):
        # plot histogram of values colored by site
        sns.histplot(data=sim_data[((sim_data['site'] != 'H0') &
                                    (sim_data['feature'] == feature) & 
                                    (sim_data['sex'] == sex) & 
                                    (sim_data['harmo'] == selected_harmo))], 
                     x='value', 
                     hue='site',
                     ax=axs[i], 
                     color='blue', 
                     alpha=0.5)
        axs[i].set_ylim(axs[0].get_ylim())
        # add vertical line for mean of each site
        site_means = controls_GAM[feature][sex_dict[sex]]['predict_vals_intervals'][0.50][20]
        axs[i].axvline(site_means, color='red', linestyle='--')
